# Multi Layer Perceptron

In [ ]:
import sys
sys.path.append('..')

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

from models.mlp import MultiLayerPerceptron
from models.activation_functions import ReLu

In [2]:
heart_disease = fetch_ucirepo(id=45)

X_raw: pd.DataFrame = heart_disease.data.features
y_raw: pd.Series = (heart_disease.data.targets['num'] > 0).astype(int)

In [3]:
mask = ~np.asarray(pd.isna(X_raw).any(axis=1))

X = X_raw[mask]
y = y_raw[mask]

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

NUM_COLS = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']

prep = ColumnTransformer(
    transformers=[
        ('stand', StandardScaler(), NUM_COLS),
    ],
    remainder="passthrough",
)

In [5]:
def log_loss_derivative(y, y_pred):
    return (-y / y_pred) + ((1 - y) / (1 - y_pred))

In [6]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y.to_numpy(), test_size=0.2, stratify=y, random_state=42
)


np.random.seed(42)

pipe = Pipeline([
    ('prep', clone(prep)),
    ('clf', MultiLayerPerceptron(
        hidden_layer_sizes=[2, 4],
        activation_function=ReLu(),
        loss_derivative=log_loss_derivative
    )),
])

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)

print(f'test accuracy: {accuracy_score(y_test, y_pred):.3f} | test f1: {f1_score(y_test, y_pred):.3f} | test precision: {precision_score(y_test, y_pred, zero_division=0)} | test recall: {recall_score(y_test, y_pred, zero_division=0)}')

test accuracy: 0.817 | test f1: 0.807 | test precision: 0.7931034482758621 | test recall: 0.8214285714285714
